In [ ]:
import os
import glob
import json
import numpy as np
import pandas as pd
from pathlib import Path
from PNW_cmap import PNW_cmap
import matplotlib.pyplot as plt
from vip_slap2_analysis.utils.utils import save_figure
from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.voltage.summary import VoltageSummary
from vip_slap2_analysis.utils.utils import normalize

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

from IPython.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

In [ ]:
%load_ext autoreload
%autoreload 2

%matplotlib notebook

In [ ]:
datapath = r"\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\ASAP7\826031\826031_2026-02-05_09-28-56\slap2\dynamic_data\dendriticVoltageExtraction\dendriticVoltageSummary-260422-111336.mat"
savepath = r"C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Data_Club\April2026\figures"

In [ ]:
from pathlib import Path
import h5py
import numpy as np

path = Path(datapath)

def _fmt_shape(shape):
    return "x".join(map(str, shape)) if len(shape) else "()"

def describe_node(node, name="/", depth=0, max_depth=2):
    indent = "  " * depth
    
    if isinstance(node, h5py.Group):
        keys = list(node.keys())
        preview = keys[:10]
        suffix = " ..." if len(keys) > 10 else ""
        print(f"{indent}{name} [Group] n_keys={len(keys)} keys={preview}{suffix}")
        
        if depth < max_depth:
            for k in keys:
                try:
                    describe_node(node[k], f"{name.rstrip('/')}/{k}", depth + 1, max_depth)
                except Exception as e:
                    print(f"{indent}  {name.rstrip('/')}/{k} [ERROR opening child: {e}]")
    
    elif isinstance(node, h5py.Dataset):
        extra = ""
        if node.dtype == h5py.ref_dtype:
            extra = " <object references>"
        print(f"{indent}{name} [Dataset] shape={_fmt_shape(node.shape)} dtype={node.dtype}{extra}")
    
    else:
        print(f"{indent}{name} [Unknown] type={type(node)}")

def deref_preview(f, ds, name, max_items=5):
    print(f"\nPreviewing refs in {name}:")
    flat = np.ravel(ds[()])
    shown = 0
    for i, ref in enumerate(flat):
        try:
            if ref is None:
                continue
            obj = f[ref]
            if isinstance(obj, h5py.Group):
                print(f"  [{i}] -> Group keys={list(obj.keys())}")
            elif isinstance(obj, h5py.Dataset):
                print(f"  [{i}] -> Dataset shape={obj.shape} dtype={obj.dtype}")
            else:
                print(f"  [{i}] -> {type(obj)}")
            shown += 1
            if shown >= max_items:
                break
        except Exception as e:
            print(f"  [{i}] -> ERROR: {e}")

print("is_hdf5:", h5py.is_hdf5(path))

with h5py.File(path, "r") as f:
    print("\nTop-level structure")
    describe_node(f, "/", max_depth=1)

    print("\nTop-level keys:", list(f.keys()))
    
    if "summary" in f:
        print("\nsummary structure")
        describe_node(f["summary"], "/summary", max_depth=2)

        print("\nsummary immediate children:")
        for k in f["summary"].keys():
            obj = f["summary"][k]
            if isinstance(obj, h5py.Dataset):
                tag = " <object refs>" if obj.dtype == h5py.ref_dtype else ""
                print(f"  {k}: Dataset shape={obj.shape} dtype={obj.dtype}{tag}")
            else:
                print(f"  {k}: Group keys={list(obj.keys())}")
    else:
        print("\nNo top-level 'summary' group found.")

    # Look for likely event/trace containers anywhere at top level
    print("\nTop-level candidate groups/datasets:")
    for k in f.keys():
        obj = f[k]
        if isinstance(obj, h5py.Dataset):
            tag = " <object refs>" if obj.dtype == h5py.ref_dtype else ""
            print(f"  {k}: Dataset shape={obj.shape} dtype={obj.dtype}{tag}")
        else:
            print(f"  {k}: Group keys={list(obj.keys())[:20]}")

    # Preview reference datasets under summary
    if "summary" in f:
        for k in f["summary"].keys():
            obj = f["summary"][k]
            if isinstance(obj, h5py.Dataset) and obj.dtype == h5py.ref_dtype:
                deref_preview(f, obj, f"/summary/{k}", max_items=5)

In [ ]:
from pathlib import PurePosixPath
import h5py

def walk_h5(node, prefix="/", max_depth=4, depth=0):
    rows = []
    if depth > max_depth:
        return rows
    
    if isinstance(node, h5py.Group):
        for k in node.keys():
            child = node[k]
            path = str(PurePosixPath(prefix) / k)
            kind = "Group" if isinstance(child, h5py.Group) else "Dataset"
            shape = getattr(child, "shape", None)
            dtype = getattr(child, "dtype", None)
            rows.append((path, kind, shape, dtype))
            rows.extend(walk_h5(child, path, max_depth=max_depth, depth=depth+1))
    return rows

with h5py.File(path, "r") as f:
    rows = walk_h5(f, "/", max_depth=4)
    
for path_i, kind, shape, dtype in rows:
    p = path_i.lower()
    if any(tok in p for tok in ["e", "roi", "rois", "trace", "traces", "global", "motion", "discard", "mask", "refplane", "userroi"]):
        print(f"{path_i:80s}  {kind:7s}  shape={shape}  dtype={dtype}")

In [ ]:
vs = VoltageSummary(datapath)

In [ ]:
dmd1_traces = vs.get_roi_traces(dmd=1,trial=1)
dmd2_traces = vs.get_roi_traces(dmd=2,trial=1)

In [ ]:
im_rate = dmd1_traces.shape[0]/1800

In [ ]:
fig,ax=plt.subplots(figsize=(6,3))

offset = 5
time = np.linspace(0,dmd1_traces.shape[0]/im_rate,dmd1_traces.shape[0])

for roi in range(dmd1_traces.shape[1]):
    
    trace = dmd1_traces[::,roi]
    trace = (trace-np.mean(trace))/np.std(trace)
    
    ax.plot(time[::],-trace[::]-offset*roi,lw=0.5)
    
    
fig.tight_layout()

In [ ]:
fig,ax=plt.subplots(figsize=(6,3))

offset = 5
time = np.linspace(0,dmd2_traces.shape[0]/im_rate,dmd2_traces.shape[0])

for roi in range(dmd2_traces.shape[1])[:]:
    
    trace = dmd2_traces[::,roi]
    trace = (trace-np.mean(trace))/np.std(trace)
    
    ax.plot(time[::],-trace[::]-offset*roi,lw=0.5)

fig.tight_layout()

In [ ]:
rois_of_interest = [0,4,10,15]
cl,cmap,cp = PNW_cmap.get_PNW_cmap('Bay',n_colors=len(rois_of_interest))
fig,ax=plt.subplots(figsize=(6,3))

ax.tick_params(axis='x', which='major', reset=True, top=False, labelsize=12)
ax.tick_params(axis='y', which='major', reset=True, right=False, labelsize=12)

offset = 5
time = np.linspace(0,dmd2_traces.shape[0]/im_rate,dmd2_traces.shape[0])
window = (600000,900000)
for i,roi in enumerate(rois_of_interest):
    
    trace = dmd2_traces[::,roi]
    trace = (trace-np.mean(trace))/np.std(trace)
    
    rolling = pd.DataFrame(trace).rolling(20,min_periods=1).mean()
    
    ax.plot(time[window[0]:window[1]]-time[window[0]],
            -rolling[window[0]:window[1]]-offset*i,
            lw=1,
           color=cp[i])
yticks = -np.arange(len(rois_of_interest)) * offset
ax.set_yticks(yticks)
ax.set_yticklabels(np.arange(len(rois_of_interest)))   
ax.set_xlabel('Time (s)')
ax.set_ylabel('Dendrite ROI')

for spine in ['top','bottom','left','right']:
    ax.spines[spine].set_linewidth(2)
ax.set_title('Dendritic voltage imaging')
fig.tight_layout()